# GTR Green Finance & Low Carbon Heating Analysis

## 📋 Overview

**Purpose**: Analyzes green finance research projects from UK Research & Innovation's Gateway to Research (GtR) using two-stage filtering to identify heating-specific research.

**Key Features**:
- Two-stage progressive filtering: `green_finance_simple` → `low_carbon_heating`
- LLM validation of research project relevance
- Time series analysis of research funding and project counts
- Automated Google Sheets export

**Expected Results**: 
- Green Finance Simple: ~60-100 research projects
- Low Carbon Heating: ~40-80 projects (typically 60-80% of green finance projects)
- Focus on UKRI research funding patterns rather than commercial investment

---

In [14]:
from discovery_utils.getters import gtr
from discovery_utils.utils import search
from discovery_utils.utils import analysis

from discovery_utils.utils.llm.batch_check import LLMProcessor, generate_relevance_check_system_message

from discovery_mission_radar import PROJECT_DIR
from pathlib import Path
#from discovery_mission_radar import VECTOR_DB_DIR

## ⚙️ Technical Requirements

**Prerequisites**:
- `discovery_utils` and `discovery_mission_radar` packages
- AWS credentials for GTR S3 access
- OpenAI API key for LLM relevance checking  
- Google Sheets API credentials

**Key Parameters**:
- `SCORE_THRESHOLD = 0.3` - Relevance score threshold
- Config files: `config_green_finance_simple.yaml`, `config_low_carbon_heating.yaml`

**Runtime**: 5-10 minutes with LLM processing, 1-2 minutes for existing data

---

## 1. Setup and Imports

In [ ]:
OUTPUT_DIR = PROJECT_DIR / 'data/green_finance/gtr/'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_DB_DIR = PROJECT_DIR / 'tmp/vector_db/'
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)
SCORE_THRESHOLD = 0.3

## 2. Configuration

In [3]:
GTR = gtr.GtrGetter(vector_db_path=VECTOR_DB_DIR)

configs = ["green_finance_simple", "low_carbon_heating"]

2025-07-31 15:48:49,743 - discovery_utils.getters.gtr - INFO - Checking for latest version of data in S3 bucket: discovery-iss
2025-07-31 15:48:49,846 - discovery_utils.getters.gtr - INFO - Latest version found: GtR_20250626


In [ ]:
import pandas as pd
import os
# define both configs
primary_config = "green_finance_simple"
secondary_config = "low_carbon_heating"

# Create config directory structure
CONFIG_DIR = PROJECT_DIR / 'discovery_mission_radar/notebooks/lch_green_finance/gtr/config/'
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

config_files = {
    primary_config: CONFIG_DIR / f"config_{primary_config}.yaml",
    secondary_config: CONFIG_DIR / f"config_{secondary_config}.yaml",
}

# Create config-specific directories with raw/ and filtered/ subdirectories
for config_name in [primary_config, secondary_config]:
    config_dir = OUTPUT_DIR / config_name
    config_dir.mkdir(exist_ok=True)
    
    raw_dir = config_dir / "raw"
    filtered_dir = config_dir / "filtered"
    raw_dir.mkdir(exist_ok=True)
    filtered_dir.mkdir(exist_ok=True)

# Check if processed data already exists
gf_csv_path = OUTPUT_DIR / primary_config / "filtered" / f"relevant_{primary_config}_llm_filtered.csv"
lc_csv_path = OUTPUT_DIR / secondary_config / "filtered" / f"relevant_{secondary_config}_llm_filtered.csv"

if gf_csv_path.exists():
    print(f"✓ Found existing {primary_config} filtered data. Loading from file...")
    df_primary_filtered = pd.read_csv(gf_csv_path)
    print(f"Loaded {len(df_primary_filtered)} {primary_config} research projects")
else:
    print(f"⚠️  No existing {primary_config} data found. Running full filtering process...")
    
    # -- STEP 1: primary green finance filter --
    
    # run keyword+vector search
    Search1 = search.SearchDataset(GTR, GTR.projects_enriched, config_files[primary_config])
    df1 = Search1.do_search()
    relevant_gf = df1.query(f"_score_avg > {SCORE_THRESHOLD}")
    gf_csv_path_temp = OUTPUT_DIR / primary_config / "raw" / f"relevant_{primary_config}.csv"
    gf_jsonl_path = OUTPUT_DIR / primary_config / "raw" / f"llm_check_MS_{primary_config}.jsonl"
    relevant_gf.to_csv(gf_csv_path_temp, index=False)
    
    # LLM‐based relevance check
    sys1 = generate_relevance_check_system_message(str(config_files[primary_config]))
    proc1 = LLMProcessor(
        output_path=str(gf_jsonl_path),
        system_message=sys1,
        session_name="mission_studio",
        output_fields=[{"name":"is_relevant","type":"str","description":"yes or no"}]
    )
    await proc1.run(dict(zip(relevant_gf['id'], relevant_gf['abstractText'])), batch_size=15, sleep_time=0.5)
    
    # Merge CSV and JSONL, filter for relevant entries, save filtered CSV
    df_csv = pd.read_csv(gf_csv_path_temp)
    df_jsonl = pd.read_json(gf_jsonl_path, lines=True)
    merged = df_csv.merge(df_jsonl[['id', 'is_relevant']], on='id', how='left')
    filtered = merged[merged['is_relevant'].str.lower() == 'yes']
    filtered.to_csv(gf_csv_path, index=False)
    rejected = merged[merged['is_relevant'].str.lower() == 'no']
    rejected.to_csv(OUTPUT_DIR / primary_config / "filtered" / f"relevant_{primary_config}_rejected.csv", index=False)
    print(f"Filtered {len(filtered)} relevant entries out of {len(df_csv)} total.")
    print(f"Filtered {len(rejected)} rejected entries out of {len(df_csv)} total.")
    
    # load the LLM‐filtered CSV
    df_primary_filtered = filtered.copy()

if lc_csv_path.exists():
    print(f"✓ Found existing {secondary_config} filtered data. Loading from file...")
    df_secondary_filtered = pd.read_csv(lc_csv_path)
    print(f"Loaded {len(df_secondary_filtered)} {secondary_config} research projects")
else:
    print(f"⚠️  No existing {secondary_config} data found. Running filtering on {primary_config} subset...")
    
    # -- STEP 2: secondary low_carbon filter on the subset --
    
    # narrow GTR.projects_enriched to only first‐stage IDs
    subset = GTR.projects_enriched[GTR.projects_enriched['id'].isin(df_primary_filtered['id'])]
    
    Search2 = search.SearchDataset(GTR, subset, config_files[secondary_config])
    df2 = Search2.do_search()
    relevant_lc = df2.query(f"_score_avg > {SCORE_THRESHOLD}")
    lc_csv_path_temp = OUTPUT_DIR / secondary_config / "raw" / f"relevant_{secondary_config}.csv"
    lc_jsonl_path = OUTPUT_DIR / secondary_config / "raw" / f"llm_check_MS_{secondary_config}.jsonl"
    relevant_lc.to_csv(lc_csv_path_temp, index=False)

    
    # LLM‐based relevance check for low carbon
    sys2 = generate_relevance_check_system_message(str(config_files[secondary_config]))
    proc2 = LLMProcessor(
        output_path=str(lc_jsonl_path),
        system_message=sys2,
        session_name="mission_studio",
        output_fields=[{"name":"is_relevant","type":"str","description":"yes or no"}]
    )
    await proc2.run(dict(zip(relevant_lc['id'], relevant_lc['abstractText'])), batch_size=15, sleep_time=0.5)
    # Merge CSV and JSONL, filter for relevant entries, save filtered CSV
    df_csv2 = pd.read_csv(lc_csv_path_temp)
    df_jsonl2 = pd.read_json(lc_jsonl_path, lines=True)
    merged2 = df_csv2.merge(df_jsonl2[['id', 'is_relevant']], on='id', how='left')
    filtered2 = merged2[merged2['is_relevant'].str.lower() == 'yes']
    filtered2.to_csv(lc_csv_path, index=False)
    rejected2 = merged2[merged2['is_relevant'].str.lower() == 'no']
    rejected2.to_csv(OUTPUT_DIR / secondary_config / "filtered" / f"relevant_{secondary_config}_rejected.csv", index=False)
    print(f"Filtered {len(filtered2)} relevant entries out of {len(df_csv2)} total.")
    print(f"Filtered {len(rejected2)} rejected entries out of {len(df_csv2)} total.")
    
    df_secondary_filtered = filtered2.copy()

print(f"\n🎯 Data Summary:")
print(f"Green Finance Simple: {len(df_primary_filtered)} research projects")
print(f"Low Carbon Heating: {len(df_secondary_filtered)} research projects")
print(f"Filtering ratio: {len(df_secondary_filtered)/len(df_primary_filtered)*100:.1f}% of green finance projects are heating-focused")

⚠️  No existing green_finance_simple data found. Running full filtering process...
2025-07-31 16:49:09,139 - discovery_utils - INFO - Keyword search found 33136 matches


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.98it/s]


2025-07-31 16:49:12,819 - discovery_utils - INFO - Vector search found 21384 matches
2025-07-31 16:49:13,839 - root - INFO - Using OpenAI
2025-07-31 16:49:13,925 - langfuse - WARNING - Langfuse client is disabled since no public_key was provided as a parameter or environment variable 'LANGFUSE_PUBLIC_KEY'. See our docs: https://langfuse.com/docs/sdk/python/low-level-sdk#initialize-client
2025-07-31 16:49:14,011 - root - INFO - Processing batch 1/70
2025-07-31 16:49:15,701 - root - INFO - Processing batch 2/70
2025-07-31 16:49:17,200 - root - INFO - Processing batch 3/70
2025-07-31 16:49:18,930 - root - INFO - Processing batch 4/70
2025-07-31 16:49:20,559 - root - INFO - Processing batch 5/70
2025-07-31 16:49:22,383 - root - INFO - Processing batch 6/70
2025-07-31 16:49:25,008 - root - INFO - Processing batch 7/70
2025-07-31 16:49:26,933 - root - INFO - Processing batch 8/70
2025-07-31 16:49:30,084 - root - INFO - Processing batch 9/70
2025-07-31 16:49:32,269 - root - INFO - Processing 

Batches: 100%|██████████| 1/1 [00:00<00:00, 85.55it/s]


2025-07-31 16:52:31,552 - discovery_utils - INFO - Vector search found 67 matches
2025-07-31 16:52:31,698 - root - INFO - Using OpenAI
2025-07-31 16:52:31,740 - langfuse - WARNING - Langfuse client is disabled since no public_key was provided as a parameter or environment variable 'LANGFUSE_PUBLIC_KEY'. See our docs: https://langfuse.com/docs/sdk/python/low-level-sdk#initialize-client
2025-07-31 16:52:31,796 - root - INFO - Processing batch 1/5
2025-07-31 16:52:33,270 - root - INFO - Processing batch 2/5
2025-07-31 16:52:35,726 - root - INFO - Processing batch 3/5
2025-07-31 16:52:38,712 - root - INFO - Processing batch 4/5
2025-07-31 16:52:40,497 - root - INFO - Processing batch 5/5
Filtered 45 relevant entries out of 61 total.
Filtered 16 rejected entries out of 61 total.

🎯 Data Summary:
Green Finance Simple: 67 research projects
Low Carbon Heating: 45 research projects
Filtering ratio: 67.2% of green finance projects are heating-focused


## 🚀 Usage Instructions

**Quick Start**: Run all cells sequentially. Existing filtered data will be loaded automatically.

**Key Parameters**:
```python
SCORE_THRESHOLD = 0.2    # Lower = more results
SHEET_TO_UPDATE = None   # Set to specific sheet name to update only that sheet
```

**Outputs**:
- **Raw/Filtered**: `data/green_finance/gtr/{config}/raw/` and `filtered/`
- **Analysis**: `data/green_finance/gtr/{config}/produce_stats/` - charts, CSVs
- **Google Sheets**: Automated upload with formatting

---

## 4. Analysis: Green Finance Simple (Stage 1)

This section analyzes the broader green finance research dataset after the first filtering stage.

In [16]:
exclude_ids = ["5F59E3B2-1ACF-4CCD-9AB1-155973E87751"] # The Active Building Centre

In [17]:
# Analysis for both configs
from pathlib import Path

# Create gtr directory for GtR analysis outputs
GTR_OUTPUT_DIR = OUTPUT_DIR

for config_name in configs:
    print(f"=== Processing {config_name} ===")
    
    # Create separate directory for this config under gtr/
    config_output_dir = GTR_OUTPUT_DIR / config_name
    config_output_dir.mkdir(exist_ok=True)
    
    # Create produce_stats subdirectory and charts subdirectory for this config
    produce_stats_dir = config_output_dir / "produce_stats"
    charts_dir = produce_stats_dir / "charts"
    chart_csvs_dir = produce_stats_dir / "chart_csvs"
    charts_dir.mkdir(parents=True, exist_ok=True)
    chart_csvs_dir.mkdir(parents=True, exist_ok=True)
    
    # Load the filtered CSV for this config from new directory structure
    csv_path = config_output_dir / "filtered" / f"relevant_{config_name}_llm_filtered.csv"
    df = (
        pd.read_csv(csv_path, parse_dates=["start", "end"], dayfirst=False)
        .query("id not in @exclude_ids")
    )

    print(f"{len(df):,} projects loaded for {config_name}")
    
    # Generate time series data
    df["year"] = df["start"].dt.year
    df["quarter"] = df["start"].dt.to_period("Q").astype(str)
    
    # Yearly summary (before imputation)
    ts_year_raw = (
        df.groupby("year", as_index=False)
          .agg(n_projects=("id", "count"),
               amount_millions=("amount", lambda s: s.sum()/1_000_000))
    )
    
    # Apply imputation to yearly data for better chart visualization
    ts_year_for_imputation = ts_year_raw.copy()
    ts_year_for_imputation['year_datetime'] = pd.to_datetime(ts_year_for_imputation['year'], format='%Y')
    
    # Apply yearly imputation using the datetime column
    ts_year_imputed = analysis.impute_empty_periods(
        ts_year_for_imputation.drop('year', axis=1), 
        time_period_col='year_datetime', 
        period='Y',  # 'Y' for annual/yearly data
        min_year=2010,  # Adjust based on GTR data range
        max_year=2025
    )
    
    # Convert back to integer year for downstream functions
    ts_year = ts_year_imputed.copy()
    ts_year['year'] = ts_year['year_datetime'].dt.year
    ts_year = ts_year.drop('year_datetime', axis=1)
    
    # Quarterly summary (before imputation)
    ts_quarter_raw = (
        df.groupby("quarter", as_index=False)
          .agg(n_projects=("id", "count"),
               amount_millions=("amount", lambda s: s.sum()/1_000_000))
    )
    
    # Apply imputation to quarterly data
    ts_quarter_for_imputation = ts_quarter_raw.copy()
    ts_quarter_for_imputation['quarter_datetime'] = pd.to_datetime(ts_quarter_for_imputation['quarter'])
    
    # Apply quarterly imputation
    ts_quarter_imputed = analysis.impute_empty_periods(
        ts_quarter_for_imputation.drop('quarter', axis=1),
        time_period_col='quarter_datetime',
        period='Q',  # 'Q' for quarterly data
        min_year=2010,
        max_year=2025
    )
    
    # Convert back to quarter string format
    ts_quarter = ts_quarter_imputed.copy()
    ts_quarter['quarter'] = ts_quarter['quarter_datetime'].dt.to_period('Q').astype(str)
    ts_quarter = ts_quarter.drop('quarter_datetime', axis=1)
    
    # Save consolidated CSV files to produce_stats directory
    # These contain all the data for this particular config with theme column added
    df.assign(theme=config_name).to_csv(produce_stats_dir / "all_projects_df.csv", index=False)
    ts_year.assign(theme=config_name).to_csv(produce_stats_dir / "all_ts_year_df.csv", index=False)
    ts_quarter.assign(theme=config_name).to_csv(produce_stats_dir / "all_ts_quarter_df.csv", index=False)
    
    # EXPORT CHART DATA AS CSVs
    # Save chart-ready data for Google Sheets visualization
    ts_year.to_csv(chart_csvs_dir / f"timeseries_year_chart_data_{config_name}.csv", index=False)
    ts_quarter.to_csv(chart_csvs_dir / f"timeseries_quarter_chart_data_{config_name}.csv", index=False)
    
    # Create and save charts to produce_stats/charts directory
    import altair as alt
    
    # Projects per year chart
    chart1 = alt.Chart(ts_year).mark_bar().encode(
        x="year:O",
        y="n_projects:Q",
        tooltip=["n_projects"]
    ).properties(
        title=f"{config_name.replace('_', ' ').title()} – number of projects per year (GtR)",
        width=600,
        height=300
    )
    chart1.save(str(charts_dir / f"{config_name}_projects_per_year.html"))
    chart1.save(str(charts_dir / f"{config_name}_projects_per_year.png"))
    
    # Funding per year chart
    chart2 = alt.Chart(ts_year).mark_bar().encode(
        x="year:O",
        y=alt.Y("amount_millions:Q", title="Amount (£ millions)"),
        tooltip=["amount_millions"]
    ).properties(
        title=f"{config_name.replace('_', ' ').title()} – funding awarded per year (GtR)",
        width=600,
        height=300
    )
    chart2.save(str(charts_dir / f"{config_name}_funding_per_year.html"))
    chart2.save(str(charts_dir / f"{config_name}_funding_per_year.png"))
    
    # Projects per quarter chart
    chart3 = alt.Chart(ts_quarter).mark_bar().encode(
        x="quarter:O",
        y="n_projects:Q",
        tooltip=["n_projects"]
    ).properties(
        title=f"{config_name.replace('_', ' ').title()} – projects per quarter (GtR)",
        width=650,
        height=300
    )
    chart3.save(str(charts_dir / f"{config_name}_projects_per_quarter.html"))
    chart3.save(str(charts_dir / f"{config_name}_projects_per_quarter.png"))
    
    print(f"Analysis complete for {config_name}")
    print(f"Total projects: {len(df)}")
    print(f"Total funding: £{df['amount'].sum()/1_000_000:.1f}M")
    print(f"Year range: {df['year'].min()}-{df['year'].max()}")
    print(f"Charts saved to: {charts_dir} (both HTML and PNG)")
    print(f"📄 Chart data exported to: {chart_csvs_dir}")
    print()

=== Processing green_finance_simple ===
67 projects loaded for green_finance_simple


/var/folders/5s/x3974l2x1hz1t5fs87qyxjgw0000gp/T/ipykernel_22194/207021858.py:68: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ts_quarter_for_imputation['quarter_datetime'] = pd.to_datetime(ts_quarter_for_imputation['quarter'])


Analysis complete for green_finance_simple
Total projects: 67
Total funding: £22.6M
Year range: 2010-2025
Charts saved to: /Users/karlis.kanders/Code/discovery_mission_radar/data/green_finance/gtr/green_finance_simple/produce_stats/charts (both HTML and PNG)
📄 Chart data exported to: /Users/karlis.kanders/Code/discovery_mission_radar/data/green_finance/gtr/green_finance_simple/produce_stats/chart_csvs

=== Processing low_carbon_heating ===
45 projects loaded for low_carbon_heating


/var/folders/5s/x3974l2x1hz1t5fs87qyxjgw0000gp/T/ipykernel_22194/207021858.py:68: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ts_quarter_for_imputation['quarter_datetime'] = pd.to_datetime(ts_quarter_for_imputation['quarter'])


Analysis complete for low_carbon_heating
Total projects: 45
Total funding: £15.7M
Year range: 2010-2024
Charts saved to: /Users/karlis.kanders/Code/discovery_mission_radar/data/green_finance/gtr/low_carbon_heating/produce_stats/charts (both HTML and PNG)
📄 Chart data exported to: /Users/karlis.kanders/Code/discovery_mission_radar/data/green_finance/gtr/low_carbon_heating/produce_stats/chart_csvs



## 5. Comprehensive Analysis: Both Filtering Stages

This section processes both configs and generates the complete analysis outputs with time series charts and funding analysis.

In [18]:
# Google Sheets Upload Configuration
from discovery_utils.utils import google

# Set your Google Sheet IDs here - one for each config
sheet_ids = {
    "green_finance_simple": "1KA3ulDizTVrMl-Ms2YSdXf0TZ5o3UAKufaqREwzaSUA",  # 
    "low_carbon_heating": "1CDqktg9tZayM7vv9JZPJGyt_TXTAfhdH-2QqKvJCch0"        # 
}

# Optionally specify a single sheet to update (e.g. "llm_filtered_projects"); set to None to update all
SHEET_TO_UPDATE = None  

# Define columns for GtR data upload
cols_projects = [
    "id", "title", "abstractText", "start", "end", "amount", "year", "quarter",
    "theme", "_score_keywords", "_score_vectors", "_score_avg", "ai_relevance_check"
]

# Process and upload MAIN DATA for each config to its respective sheet
for config_name in configs:
    print(f"Processing {config_name} for Google Sheets upload (MAIN DATA ONLY)...")
    
    # Get the sheet ID for this config
    sheet_id = sheet_ids[config_name]
    
    # Load the CSV files for this config from new directory structure
    config_output_dir = GTR_OUTPUT_DIR / config_name
    produce_stats_dir = config_output_dir / "produce_stats"
    
    try:
        # Read the main data files from produce_stats directory
        all_projects_df = pd.read_csv(produce_stats_dir / "all_projects_df.csv")
        all_ts_year_df = pd.read_csv(produce_stats_dir / "all_ts_year_df.csv")
        all_ts_quarter_df = pd.read_csv(produce_stats_dir / "all_ts_quarter_df.csv")
        
        # Load relevant data from raw and filtered directories
        relevant_df = pd.read_csv(config_output_dir / "raw" / f"relevant_{config_name}.csv")
        relevant_check_df = pd.read_json(config_output_dir / "raw" / f"llm_check_MS_{config_name}.jsonl", lines=True)
        relevant_checked_df = relevant_df.merge(
            relevant_check_df[['id', 'is_relevant']], on='id', how='left'
        ).rename(columns={"is_relevant": "ai_relevance_check"})
        
        # Filter data to match available columns
        available_cols = [col for col in cols_projects if col in relevant_checked_df.columns]
        _all_projects = relevant_checked_df[available_cols]
        
        # ALSO export only the LLM-approved rows
        _llm_only = _all_projects.query("ai_relevance_check == 'yes'")
        
        print(f"Data prepared for {config_name}:")
        print(f"  - Projects: {len(_all_projects)} (pre-LLM), {len(_llm_only)} (LLM-approved)")
        print(f"  - Time series (yearly): {len(all_ts_year_df)} data points")
        print(f"  - Time series (quarterly): {len(all_ts_quarter_df)} data points")
        
        # Upload MAIN DATA to Google Sheets with clean sheet names (no config suffix since each has its own sheet)
        sheet_names = {
            "gtr_projects": _all_projects,
            "llm_filtered_projects": _llm_only,
            "gtr_timeseries_year": all_ts_year_df,
            "gtr_timeseries_quarter": all_ts_quarter_df
        }
        
        # Only upload non-empty dataframes (and optionally limit to one sheet)
        if SHEET_TO_UPDATE:
            upload_data = {name: df for name, df in sheet_names.items()
                           if name == SHEET_TO_UPDATE and not df.empty}
        else:
            upload_data = {name: df for name, df in sheet_names.items() if not df.empty}
        
        print(f"  - Preparing to upload {len(upload_data)} MAIN DATA sheets")
        
        if upload_data:
            google.upload_data_to_gsheet(sheet_id, upload_data)
            print(f"✓ Uploaded {len(upload_data)} MAIN DATA sheets for {config_name} to Google Sheet: {sheet_id}")
            
            # Apply formatting to key sheets with error handling
            try:
                google.format_gsheet(sheet_id, "gtr_projects", freeze_cols=3)
                google.format_gsheet(sheet_id, "llm_filtered_projects", freeze_cols=3)
                        
            except Exception as e:
                print(f"  - Warning: Could not apply formatting: {e}")
            
        else:
            print(f"⚠️  No data to upload for {config_name}")
            
    except FileNotFoundError as e:
        print(f"⚠️  Could not find data files for {config_name}: {e}")
        print("Make sure you've run the analysis sections above first.")
    except Exception as e:
        print(f"❌ Error uploading {config_name} to Google Sheets: {e}")

print("\\n🎯 Main data Google Sheets upload complete!")

Processing green_finance_simple for Google Sheets upload (MAIN DATA ONLY)...
Data prepared for green_finance_simple:
  - Projects: 1044 (pre-LLM), 67 (LLM-approved)
  - Time series (yearly): 16 data points
  - Time series (quarterly): 64 data points
  - Preparing to upload 4 MAIN DATA sheets
2025-07-31 16:59:35,371 - root - INFO - Connected to Google Sheet: green finance
2025-07-31 16:59:36,210 - root - INFO - Uploading DataFrame to sheet: gtr_projects


/Users/karlis.kanders/Code/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Code/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)


2025-07-31 16:59:59,572 - root - INFO - Connected to Google Sheet: green finance
2025-07-31 17:00:00,303 - root - INFO - Uploading DataFrame to sheet: llm_filtered_projects


/Users/karlis.kanders/Code/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Code/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)


2025-07-31 17:00:12,559 - root - INFO - Connected to Google Sheet: green finance
2025-07-31 17:00:13,684 - root - INFO - Uploading DataFrame to sheet: gtr_timeseries_year


...

2025-07-31 17:01:12,698 - root - INFO - Connected to Google Sheet: LCH green finance
2025-07-31 17:01:14,456 - root - INFO - Uploading DataFrame to sheet: gtr_timeseries_year


/Users/karlis.kanders/Code/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Code/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)


2025-07-31 17:01:27,493 - root - INFO - Connected to Google Sheet: LCH green finance
2025-07-31 17:01:28,188 - root - INFO - Uploading DataFrame to sheet: gtr_timeseries_quarter


/Users/karlis.kanders/Code/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Code/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)


2025-07-31 17:01:38,691 - root - INFO - Upload completed successfully.
✓ Uploaded 4 MAIN DATA sheets for low_carbon_heating to Google Sheet: 1CDqktg9tZayM7vv9JZPJGyt_TXTAfhdH-2QqKvJCch0
2025-07-31 17:01:39,465 - root - INFO - Connected to Google Sheet: LCH green finance
2025-07-31 17:01:44,475 - root - INFO - Connected to Google Sheet: LCH green finance
\n🎯 Main data Google Sheets upload complete!


In [19]:
# Google Sheets Upload - CHART DATA ONLY
# Run this cell separately to upload just the chart data when needed

# Use the same sheet IDs as above
chart_sheet_ids = {
    "green_finance_simple": "1KA3ulDizTVrMl-Ms2YSdXf0TZ5o3UAKufaqREwzaSUA",
    "low_carbon_heating": "1CDqktg9tZayM7vv9JZPJGyt_TXTAfhdH-2QqKvJCch0"
}

# Optionally specify a single chart to update, or None for all charts
CHART_TO_UPDATE = None  # e.g. "chart_timeseries_year" or None for all

print("🔥 Starting CHART DATA upload...")

# Process and upload CHART DATA for each config
for config_name in configs:
    print(f"Processing {config_name} for Chart Data upload...")
    
    # Get the sheet ID for this config
    sheet_id = chart_sheet_ids[config_name]
    
    # Load chart CSV data files
    config_output_dir = GTR_OUTPUT_DIR / config_name
    produce_stats_dir = config_output_dir / "produce_stats"
    chart_csvs_dir = produce_stats_dir / "chart_csvs"
    
    try:
        chart_data = {}
        chart_files = {
            "chart_timeseries_year": f"timeseries_year_chart_data_{config_name}.csv",
            "chart_timeseries_quarter": f"timeseries_quarter_chart_data_{config_name}.csv"
        }
        
        # Load chart data with error handling
        for tab_name, filename in chart_files.items():
            chart_file_path = chart_csvs_dir / filename
            if chart_file_path.exists():
                try:
                    chart_df = pd.read_csv(chart_file_path)
                    if not chart_df.empty:
                        chart_data[tab_name] = chart_df
                        print(f"  ✓ Loaded {tab_name}: {len(chart_df)} rows")
                    else:
                        print(f"  ⚠️  Chart file is empty: {filename}")
                except Exception as e:
                    print(f"  ❌ Error loading chart file {filename}: {e}")
            else:
                print(f"  ⚠️  Chart file not found: {filename}")
        
        # Filter to specific chart if requested
        if CHART_TO_UPDATE:
            if CHART_TO_UPDATE in chart_data:
                chart_data = {CHART_TO_UPDATE: chart_data[CHART_TO_UPDATE]}
                print(f"  - Uploading only: {CHART_TO_UPDATE}")
            else:
                print(f"  ⚠️  Requested chart {CHART_TO_UPDATE} not found")
                continue
        
        print(f"  - Chart datasets ready: {len(chart_data)}")
        
        if chart_data:
            # Upload chart data to Google Sheets
            google.upload_data_to_gsheet(sheet_id, chart_data)
            print(f"✓ Uploaded {len(chart_data)} CHART sheets for {config_name} to Google Sheet: {sheet_id}")
            
            # Apply formatting to chart sheets
            try:
                for chart_tab in chart_data.keys():
                    google.format_gsheet(sheet_id, chart_tab, freeze_cols=1)
                print(f"  ✓ Applied formatting to {len(chart_data)} chart sheets")
                        
            except Exception as e:
                print(f"  - Warning: Could not apply formatting to charts: {e}")
            
        else:
            print(f"⚠️  No chart data to upload for {config_name}")
            
    except Exception as e:
        print(f"❌ Error uploading chart data for {config_name}: {e}")
        import traceback
        traceback.print_exc()

print("\\n🎯 Chart data Google Sheets upload complete!")

🔥 Starting CHART DATA upload...
Processing green_finance_simple for Chart Data upload...
  ✓ Loaded chart_timeseries_year: 16 rows
  ✓ Loaded chart_timeseries_quarter: 64 rows
  - Chart datasets ready: 2
2025-07-31 17:01:48,384 - root - INFO - Connected to Google Sheet: green finance
2025-07-31 17:01:49,692 - root - INFO - Uploading DataFrame to sheet: chart_timeseries_year


/Users/karlis.kanders/Code/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Code/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)


2025-07-31 17:02:04,644 - root - INFO - Connected to Google Sheet: green finance
2025-07-31 17:02:05,865 - root - INFO - Uploading DataFrame to sheet: chart_timeseries_quarter


/Users/karlis.kanders/Code/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Code/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)


2025-07-31 17:02:15,774 - root - INFO - Upload completed successfully.
✓ Uploaded 2 CHART sheets for green_finance_simple to Google Sheet: 1KA3ulDizTVrMl-Ms2YSdXf0TZ5o3UAKufaqREwzaSUA
2025-07-31 17:02:16,406 - root - INFO - Connected to Google Sheet: green finance
2025-07-31 17:02:21,061 - root - INFO - Connected to Google Sheet: green finance
  ✓ Applied formatting to 2 chart sheets
Processing low_carbon_heating for Chart Data upload...
  ✓ Loaded chart_timeseries_year: 16 rows
  ✓ Loaded chart_timeseries_quarter: 64 rows
  - Chart datasets ready: 2
2025-07-31 17:02:24,885 - root - INFO - Connected to Google Sheet: LCH green finance
2025-07-31 17:02:26,742 - root - INFO - Uploading DataFrame to sheet: chart_timeseries_year


/Users/karlis.kanders/Code/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Code/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)


2025-07-31 17:02:38,588 - root - INFO - Connected to Google Sheet: LCH green finance
2025-07-31 17:02:40,304 - root - INFO - Uploading DataFrame to sheet: chart_timeseries_quarter


/Users/karlis.kanders/Code/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Code/discovery_mission_radar/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)


2025-07-31 17:02:51,248 - root - INFO - Upload completed successfully.
✓ Uploaded 2 CHART sheets for low_carbon_heating to Google Sheet: 1CDqktg9tZayM7vv9JZPJGyt_TXTAfhdH-2QqKvJCch0
2025-07-31 17:02:52,283 - root - INFO - Connected to Google Sheet: LCH green finance
2025-07-31 17:02:56,257 - root - INFO - Connected to Google Sheet: LCH green finance
  ✓ Applied formatting to 2 chart sheets
\n🎯 Chart data Google Sheets upload complete!


---

## 🔧 Troubleshooting

**Expected Results**:
- Green Finance Simple: 60-100 projects
- Low Carbon Heating: 40-80 projects (60-80% filtering ratio)

**Common Issues**:
- Missing config files → Ensure YAML files exist in `config/` directory
- LLM timeouts → Reduce batch_size from 15 to 10
- Google Sheets errors → Verify service account permissions

**Performance Tips**:
- Use existing filtered data (don't delete `filtered/` dirs)
- Set `SHEET_TO_UPDATE` for partial uploads
- Check intermediate files in `raw/` directories for debugging